Databricks notebook: nb_gold_fact_charging_session
Source : ev_catalog.silver.sl_sessions  +  sl_payments  +  sl_tariffs
Sink   : ev_catalog.gold.fact_charging_session  (Delta, external)
Grain  : One row per charging session

Silver sl_sessions columns (all numeric already typed by Silver):
  session_id, vehicle_id, station_id, customer_id,
  duration_min (LONG), energy_kwh (DOUBLE), cost_aud (DOUBLE),
  excl_gst_aud (DOUBLE), gst_amount_aud (DOUBLE), peak_power_kw (DOUBLE),
  connector_type, session_status, payment_id,
  sla_breach_flag (BOOLEAN), late_event_flag (BOOLEAN),
  started_at_utc, started_at_aest, ended_at_utc, ended_at_aest,
  created_at_utc, created_at_aest, updated_at_utc, updated_at_aest

Silver sl_payments columns:
  payment_id, session_id, customer_id, gateway,
  amount_aud (DOUBLE), excl_gst_aud (DOUBLE), gst_amount_aud (DOUBLE),
  payment_mode, status, billing_anomaly_flag (BOOLEAN),
  processed_at_utc, processed_at_aest

Silver sl_tariffs: tariff_id, rate_per_kwh (DOUBLE), is_current (BOOLEAN)

In [ ]:
# ── Cell 1: imports and config ────────────────────────────────────────────────
from pyspark.sql import functions as F
from delta.tables import DeltaTable

GOLD_TABLE = "ev_catalog.gold.fact_charging_session"
GOLD_PATH  = "abfss://gold@evdatalakedev2.dfs.core.windows.net/facts/fact_charging_session/"

dbutils.widgets.text("p_run_date", "", "Run date YYYY-MM-DD (blank = full load)")
dbutils.widgets.text("p_run_type", "", "Run type full/incremental (blank = full load)")
run_date = dbutils.widgets.get("p_run_date").strip()
run_type = dbutils.widgets.get("p_run_type").strip()

In [ ]:
# ── Cell 2: read Silver sources ────────────────────────────────────────────────
df_sessions = spark.table("ev_catalog.silver.sl_sessions")
df_payments = spark.table("ev_catalog.silver.sl_payments")
df_tariffs  = spark.table("ev_catalog.silver.sl_tariffs")

# Incremental filter: sl_sessions has sl_batch_id from Silver MERGE
if run_date and run_type == 'incremental':
    df_sessions = df_sessions.filter(F.col("sl_batch_id") == run_date)
    print(f"Incremental: filtering sessions for batch_id = {run_date}")

In [ ]:
# ── Cell 3: resolve current tariff rate ───────────────────────────────────────
# rate_per_kwh is already DOUBLE in Silver. Collect one current rate
# to compute expected_amount_aud = energy_kwh * rate_per_kwh.
df_current_tariff = (
    df_tariffs
    .filter(F.col("is_current") == True)
    .select("tariff_id", "rate_per_kwh")
    .limit(1)
)

current_rate = df_current_tariff.collect()
rate_per_kwh = float(current_rate[0]["rate_per_kwh"]) if current_rate else 0.0
tariff_key   = current_rate[0]["tariff_id"]           if current_rate else "UNKNOWN"
print(f"Current tariff: {tariff_key}  rate={rate_per_kwh}")

In [ ]:
# ── Cell 4: build fact ────────────────────────────────────────────────────────
# All numeric fields (energy_kwh, cost_aud, peak_power_kw, excl_gst_aud,
# gst_amount_aud, amount_aud) are already correctly typed by Silver.
# sla_breach_flag, late_event_flag, billing_anomaly_flag already BOOLEAN in Silver.
# started_at_utc/aest exist; no raw started_at string.
# Join payments via sessions.payment_id = payments.payment_id.

df_fact = (
    df_sessions.alias("s")
    .join(
        df_payments.alias("p").select(
            "payment_id",
            "amount_aud",
            "gst_amount_aud",
            "excl_gst_aud",
            "status",
            "gateway",
            "payment_mode",
            "billing_anomaly_flag",
        ),
        F.col("s.payment_id") == F.col("p.payment_id"),
        "left"
    )
    .select(
        # Keys
        F.col("s.session_id")                           .alias("session_key"),
        F.col("s.customer_id")                          .alias("customer_key"),
        F.col("s.station_id")                           .alias("station_key"),
        F.lit(tariff_key)                               .alias("tariff_key"),
        # Time — use started_at_aest for local-time reporting
        F.date_format(F.col("s.started_at_aest"), "yyyyMMddHH").cast("long").alias("time_key"),
        F.to_date(F.col("s.started_at_aest"))           .alias("session_date"),
        # Session measures (already typed DOUBLE/LONG by Silver)
        F.col("s.energy_kwh"),
        F.col("s.duration_min"),
        F.col("s.peak_power_kw"),
        F.col("s.cost_aud"),
        F.col("s.excl_gst_aud")                        .alias("session_excl_gst_aud"),
        F.col("s.gst_amount_aud")                      .alias("session_gst_aud"),
        # Expected amount from current tariff
        F.round(F.col("s.energy_kwh") * F.lit(rate_per_kwh), 2).alias("expected_amount_aud"),
        # Payment measures
        F.col("p.amount_aud")                           .alias("billed_amount_aud"),
        F.col("p.gst_amount_aud")                      .alias("payment_gst_aud"),
        F.col("p.excl_gst_aud")                        .alias("payment_excl_gst_aud"),
        F.col("p.status")                               .alias("payment_status"),
        F.col("p.gateway")                              .alias("payment_gateway"),
        F.col("p.payment_mode"),
        F.col("p.billing_anomaly_flag"),
        # Reconciliation
        F.when(F.col("p.amount_aud").isNull(), "UNPAID")
         .when(
             F.abs(F.col("p.amount_aud") - F.round(F.col("s.energy_kwh") * F.lit(rate_per_kwh), 2)) < 0.01,
             "MATCH"
         )
         .otherwise("MISMATCH")                        .alias("reconciliation_status"),
        # Quality flags (already BOOLEAN from Silver)
        F.col("s.sla_breach_flag"),
        F.col("s.late_event_flag"),
        # Session attributes
        F.col("s.session_status"),
        F.col("s.connector_type"),
        # Metadata
        F.current_timestamp()                           .alias("gold_updated_at"),
        F.lit(run_date if run_date else "full")         .alias("gold_batch_id"),
    )
)

total = df_fact.count()
print(f"Fact rows: {total}")
if total == 0:
    print("No sessions to process. Exiting.")
    dbutils.notebook.exit("NO_DATA")

df_fact.show(3, truncate=80)

In [ ]:
# ── Cell 5: reconciliation summary (before write) ─────────────────────────────
df_fact.groupBy("reconciliation_status").count().display()

In [ ]:
# ── Cell 6: CREATE TABLE IF NOT EXISTS ────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_TABLE} (
    session_key            STRING    NOT NULL,
    customer_key           STRING,
    station_key            STRING,
    tariff_key             STRING,
    time_key               LONG,
    session_date           DATE,
    energy_kwh             DOUBLE,
    duration_min           LONG,
    peak_power_kw          DOUBLE,
    cost_aud               DOUBLE,
    session_excl_gst_aud   DOUBLE,
    session_gst_aud        DOUBLE,
    expected_amount_aud    DOUBLE,
    billed_amount_aud      DOUBLE,
    payment_gst_aud        DOUBLE,
    payment_excl_gst_aud   DOUBLE,
    payment_status         STRING,
    payment_gateway        STRING,
    payment_mode           STRING,
    billing_anomaly_flag   BOOLEAN,
    reconciliation_status  STRING,
    sla_breach_flag        BOOLEAN,
    late_event_flag        BOOLEAN,
    session_status         STRING,
    connector_type         STRING,
    gold_updated_at        TIMESTAMP,
    gold_batch_id          STRING
)
USING DELTA
LOCATION '{GOLD_PATH}'
""")

In [ ]:
# ── Cell 7: MERGE ─────────────────────────────────────────────────────────────
tgt = DeltaTable.forName(spark, GOLD_TABLE)
(
    tgt.alias("t")
       .merge(df_fact.alias("s"), "t.session_key = s.session_key")
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll()
       .execute()
)
print(f"MERGE done: {GOLD_TABLE}")

In [ ]:
# ── Cell 8: post-load reconciliation breakdown ────────────────────────────────
spark.sql(f"""
SELECT
    reconciliation_status,
    COUNT(*)                               AS session_count,
    ROUND(SUM(expected_amount_aud), 2)     AS total_expected,
    ROUND(SUM(billed_amount_aud),   2)     AS total_billed,
    ROUND(SUM(billed_amount_aud) - SUM(expected_amount_aud), 2) AS variance_aud
FROM {GOLD_TABLE}
GROUP BY reconciliation_status
ORDER BY reconciliation_status
""").display()